# Compare benchmark results

Allows comparing the benchmark results of each index implementation, within the same benchmark report.

To execute the benchmarks and generate a report, run:
  * `./gradlew benchmark` - standard benchmarks; faster to run but has higher score error
  * `./gradlew extendedBenchmark` - extended benchmarks; takes much longer but improves result accuracy

> Based on https://github.com/Kotlin/kotlinx-benchmark/blob/master/examples/simple-benchmark-analysis.ipynb

In [ ]:
%use serialization, dataframe, kandy


In [ ]:
@Serializable
data class Benchmark(
    val benchmark: String,
    val mode: String,
    val warmupIterations: Int,
    val warmupTime: String,
    val measurementIterations: Int,
    val measurementTime: String,
    val primaryMetric: Metric,
    val secondaryMetrics: Map<String, Metric>,
    val params: JsonObject? = null
)

@Serializable
data class Metric(
    val score: Double,
    val scoreError: Double,
    val scoreConfidence: List<Double>,
    val scorePercentiles: Map<String, Double>,
    val scoreUnit: String,
    val rawData: List<List<Double>>,
)

@Serializable
data class BenchmarkGroup(
    val name: String,
    val params: String,
    val score: Double,
    val error: Double,
    val unit: String
)

In [ ]:
import java.nio.file.Files
import java.nio.file.attribute.BasicFileAttributes
import kotlin.io.path.exists
import kotlin.io.path.forEachDirectoryEntry
import kotlin.io.path.isDirectory
import kotlin.io.path.listDirectoryEntries
import kotlin.io.path.readText

val benchmarksDirectory = notebook.workingDir.resolve("../build/reports/benchmarks").normalize()
val mainReportsDirectory = benchmarksDirectory.resolve("main")
val extendedReportsDirectory = benchmarksDirectory.resolve("extended")
val reportFiles = (mainReportsDirectory.listDirectoryEntries() + extendedReportsDirectory.listDirectoryEntries())
    .filter { it.isDirectory() && it.listDirectoryEntries().isNotEmpty() }
    .sortedByDescending { dir -> Files.readAttributes(dir, BasicFileAttributes::class.java).creationTime() }
    .map { it.resolve("benchmarks.json") }

reportFiles.map { it.normalize().parent }.withIndex().map { (index, report) ->
    Triple(index, report.parent.fileName, report.fileName)
}.toDataFrame().rename("first" to " ", "second" to "type", "third" to "timestamp")

In [ ]:
// select which report file should be used for the comparison
// default: newest
val selectedReportFile = reportFiles[0]

In [ ]:
import kotlinx.serialization.json.encodeToJsonElement

val json = Json { ignoreUnknownKeys = true }

val selectedReport = json.decodeFromString<List<Benchmark>>(selectedReportFile.readText())

In [ ]:
val groupedResults = selectedReport
    .groupBy { it.benchmark }
    .mapValues { group ->
        val benchmarks = group.value.map { benchmark ->
            val paramInfo = benchmark.params?.entries.orEmpty()
                .sortedBy { it.key }
                .joinToString(",") { "${it.key}=${it.value.jsonPrimitive.content}" }
            val name = benchmark.benchmark
            BenchmarkGroup(
                name = name,
                params = paramInfo,
                score = benchmark.primaryMetric.score,
                error = benchmark.primaryMetric.scoreError,
                unit = benchmark.primaryMetric.scoreUnit
            )
        }
        benchmarks.toDataFrame()
    }

In [ ]:
val data = groupedResults.mapValues {
    it.value
        .add("errorMin") { it.getValue<Double>("score") - it.getValue<Double>("error") }
        .add("errorMax") { it.getValue<Double>("score") + it.getValue<Double>("error") }
        .insert("label") {
            if (!it.getValue<String>("params").isBlank()) {
                it.getValue<String>("params").replace(",", "\n")
            } else {
                it.getValue<String>("name").substringAfterLast(".")
            }
        }.at(0)
        .remove("name", "params")
}

In [ ]:
import org.jetbrains.letsPlot.themes.margin

val reportType = selectedReportFile.parent.parent.fileName.toString()
val reportTimestamp = selectedReportFile.parent.fileName.toString()

DISPLAY(HTML("<h3 >Report - ${reportTimestamp} [${reportType}]</h3>"))

data.forEach { (name, df) ->
    val plot = df.plot {
        bars {
            x("label") {
                axis.name = ""
            }
            y("score") {
                axis.breaks(format = ".3s")
            }
        }
        errorBars {
            x("label")
            y("score")
            yMin("errorMin")
            yMax("errorMax")
        }
        coordinatesTransformation = CoordinatesTransformation.cartesianFlipped()
        layout {
            yAxisLabel = df.first().getValue<String>("unit")
            style {
                global {
                    title {
                        margin(10.0, 0.0)
                    }
                    text {
                        fontFamily = FontFamily.MONO
                    }
                }
            }
            size = 800 to ((50 * df.size().nrow))
        }
    }
    DISPLAY(HTML("<h5 >$name</h5>"))
    DISPLAY(plot)
}